# 📊 Monster Breach — Battle Dashboard

Live view of the war on corruption: boss HP, monsters defeated, crystals saved, and the
ranger leaderboard — all read from the **`BattleEvents`** KQL table.

Run all cells after you have attempted a few levels in **`MonsterBreach_Judge`**.

## Step 1 — Connect to BattleData

In [ ]:
import requests
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, Markdown

WORKSPACE_ID = None
try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")
except Exception:
    pass

EH_NAME = "BattleData"
EH_TABLE = "BattleEvents"

def _token(resource):
    import notebookutils
    return notebookutils.credentials.getToken(resource)

def _fabric_get(url):
    r = requests.get(url, headers={"Authorization": f"Bearer {_token('pbi')}"}, timeout=60)
    r.raise_for_status()
    return r.json()

_dbs = _fabric_get(f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/kqlDatabases").get("value", [])
_t = next(d for d in _dbs if d["displayName"] == EH_NAME)
_info = _fabric_get(f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/kqlDatabases/{_t['id']}")
QSI = _info["properties"]["queryServiceUri"]

def kql(q):
    r = requests.post(f"{QSI}/v1/rest/query",
                      headers={"Authorization": f"Bearer {_token('kusto')}", "Content-Type": "application/json"},
                      json={"db": EH_NAME, "csl": q}, timeout=30)
    r.raise_for_status()
    tbl = r.json()["Tables"][0]
    cols = [c["ColumnName"] for c in tbl["Columns"]]
    return [dict(zip(cols, row)) for row in tbl["Rows"]]

print("✅ Connected to BattleData.")

## Step 2 — Boss HP (corruption remaining)

In [ ]:
# Each level cleared deals damage. 4 levels + boss = 5 hits to zero HP.
rows = kql(f"{EH_TABLE} | where EventType in ('LevelComplete','BossDefeated') | summarize Cleared = dcount(Level)")
cleared = rows[0]["Cleared"] if rows else 0
hp = max(0, 100 - int(cleared) * 20)
fig = go.Figure(go.Indicator(
    mode="gauge+number", value=hp,
    title={"text": "👑 Corruption King HP"},
    gauge={"axis": {"range": [0, 100]},
           "bar": {"color": "crimson"},
           "steps": [{"range": [0, 33], "color": "#d5f5e3"},
                     {"range": [33, 66], "color": "#fdebd0"},
                     {"range": [66, 100], "color": "#fadbd8"}]}))
fig.show()

## Step 3 — Monsters defeated

In [ ]:
rows = kql(f"{EH_TABLE} | where EventType in ('LevelComplete','BossDefeated') | distinct Level, Monster | order by Level asc")
if rows:
    md = ["| Level | Monster | Status |", "|---|---|---|"]
    for r in rows:
        md.append(f"| {r['Level']} | {r['Monster']} | ✅ defeated |")
    display(Markdown("\n".join(md)))
else:
    display(Markdown("*No monsters defeated yet — go build a pipeline!*"))

## Step 4 — Crystals saved over time

In [ ]:
rows = kql(f"{EH_TABLE} | where EventType == 'LevelComplete' or EventType == 'BossDefeated' | summarize Saved = sum(CrystalsSaved) by bin(Timestamp, 1m) | order by Timestamp asc")
if rows:
    xs = [r["Timestamp"] for r in rows]
    ys = [r["Saved"] for r in rows]
    fig = px.area(x=xs, y=ys, labels={"x": "Time", "y": "Crystals saved"}, title="💎 Crystals saved")
    fig.show()
else:
    display(Markdown("*No crystals saved yet.*"))

## Step 5 — Ranger leaderboard

In [ ]:
rows = kql(f"{EH_TABLE} | where EventType in ('LevelComplete','BossDefeated') | summarize Score = sum(Score), Levels = dcount(Level) by PlayerId | order by Score desc")
if rows:
    md = ["| Rank | Ranger | Levels | Score |", "|---|---|---|---|"]
    for i, r in enumerate(rows, 1):
        md.append(f"| {i} | {r['PlayerId']} | {r['Levels']} | {int(r['Score'])} |")
    display(Markdown("\n".join(md)))
else:
    display(Markdown("*Leaderboard empty.*"))